[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/paramagnetic_salt/paramagnetic_salt.ipynb)

# Adiabatic Demagnetization

This example simulates adiabatic demagnetization across a paramagnetic crystal slab cycled as a magnetic refrigerator between a warm bath at one end and a millikelvin load at the other. A paramagnetic salt is modelled as non-interacting two-level spins in a crystal lattice that holds a small heat capacity. Applying a magnetic field aligns the spins and lowers their mixing entropy; when magnetized in contact with a bath the salt rejects that entropy as heat, and when isolated and demagnetized the disordering spins draw entropy back out of the lattice and cool the crystal. Just as an ideal polymer chain stores internal energy independent of stretch so that tension arises from entropy, an ideal paramagnet stores internal energy in its lattice alone so that its equilibrium magnetic field arises from spin entropy. Across a finite crystal, thermal conduction takes time to cross the lattice, so rapid field sweeps warm or cool the interior ahead of the linked reservoir and launch temperature fronts inward from the contact end. Debye and Giauque proposed the method independently in 1926 and 1927, and in 1933 Giauque and MacDougall cooled gadolinium sulfate below a quarter of a kelvin, work honoured by Giauque's 1949 Nobel Prize in Chemistry.

In planar projective spacetime joined with a metric-free entropy direction, each dual cell combines its spin-turning magnetization patch `magnet_plane.dual()` and entropy quantity `s.dual()` into one intrinsic state, while the magnetic field plane and temperature along `s` combine into one extrinsic potential whose pairing `potentials & change` expresses the first law without cross-talk between magnetic work `field & spin` and heat. Differentiating the lattice energy over the state yields `temperature * (s - spin_slope)`, identifying the equilibrium field directly with temperature times the spin-entropy slope. Differentiating a second time forms the thermodynamic stiffness as a symmetric sum of two dyads that carries the magnetocaloric Maxwell relation directly, and its per-vertex inverse couples with the simplex-gradient conduction pullback `gradient.adjugate()(medium(gradient))` in a single implicit heat-balance solve per step, mirroring the entropic cycle of the [rubber band](../rubber_band/).

In thermodynamic partial-derivative notation, as in [Debye](https://archive.org/download/sim_annalen-der-physik_1926_81_25/sim_annalen-der-physik_1926_81_25.pdf#page=285) and in the work of Giauque recounted in [his National Academy biographical memoir](https://nap.nationalacademies.org/resource/biomems/wgiauque.html), writing the first law as $dU = T\,dS + B\,dM$ tracks state derivatives through held-variable subscripts such as $(\partial T/\partial M)_S = (\partial B/\partial S)_M$, changing independent variables from $(S, M)$ to $(T, B)$ expands the held-field heat capacity via Jacobian chains as $C_B = T(\partial S/\partial T)_B = C_M - T(\partial M/\partial T)_B(\partial B/\partial T)_M$ and the isothermal magnetocaloric coefficient as $(\partial S/\partial B)_T = (\partial M/\partial T)_B$, and coupling them to heat flow writes a separate scalar transport equation $C_B\,\partial_t T = \nabla\cdot(\kappa\nabla T) - T(\partial M/\partial T)_B\,\partial_t B$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack, where
from numga.sparse import spdiag
from examples.thermodynamics.paramagnetic_salt import render, scenarios
from examples.util.animation import show_states

np.set_printoptions(precision=3, suppress=True)

# Planar projective spacetime with null weight w joined with a metric-free direction s for entropy.
space = Algebra("x+y+")
ga = Algebra("x+y+t-w0") * Algebra("s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# Spatial planes along and across the crystal slab.
Plane = ga.gatype.vector().derive.within(space)                               # Vector+
# Magnetic field plane and intrinsic spin-turning patch.
Magnetic = ga.gatype.bivector().derive.within(space)                          # Bivector+
Spin = Magnetic.derive.dual                                                   # Antibivector-
# Thermal plane along s for temperature and its dual entropy quantity.
Thermal = ga.gatype.vector().derive.within(Algebra("s0"))                     # Vector+
Quantity = Thermal.derive.dual                                                # Antivector-
# Combined thermodynamic state, conjugate potential, and energy Hessian stiffness.
State = ga.gatype(Spin + Quantity)                                            # Antibivector- + Antivector-
Potential = ga.gatype(Magnetic + Thermal)                                     # Bivector+ + Vector+
Stiffness = ga.gatype((Potential, State))                                     # Potential+ <- State-
# Unit turning plane, intrinsic spin patch, entropy quantity, time direction, and unit heat antiscalar.
magnet_plane = mv.xy                                                          # [] Magnetic+
spin = magnet_plane.dual()                                                    # [] Spin-
entropy = mv.s.dual()                                                         # [] Quantity-
time = mv.t.dual()                                                            # [] Point-
heat = mv.scalar([1]).dual()                                                  # [] Antiscalar-
crystal_axes = stack((mv.x.dual(), mv.y.dual()))                              # [2] Point-

## Paramagnetic equation of state

Two-level spins carry mixing entropy determined solely by their aligned and opposed population fractions, falling from `ln 2` per spin at zero magnetization to zero at saturation. The crystal lattice holds the remaining entropy at constant heat capacity so its temperature grows exponentially with lattice entropy, and because internal energy resides in lattice heat alone, differentiating energy over the state yields `temperature * (s - spin_slope)`. Pairing that potential with unit spin gives the spin-half hyperbolic-tangent equation of state, reducing at weak fields to Curie's linear law. Temperatures are measured in kelvin, fields in tesla, entropy in Boltzmann's constant per spin referenced to the bath temperature, and magnetization in kelvin per tesla per spin with electron saturation moment 0.672 kelvin per tesla.

In [ ]:
moment, heat_capacity, reference = scenarios.moment, scenarios.heat_capacity, scenarios.reference
n_shares, largest_share = scenarios.n_shares, scenarios.largest_share

In [ ]:
def spin_entropy(shares: Scalar) -> Scalar:
    """Two-level mixing entropy per spin (`[...] Scalar+`) falls from `ln 2` at zero magnetization to zero as `shares`
    (`[...] Scalar+`) approaches saturation."""
    aligned_share, opposed_share = (1 + shares) / 2, (1 - shares) / 2         # [...] Scalar+, Scalar+
    return -(aligned_share * aligned_share.log() + opposed_share * opposed_share.log())


def potentials(state: State) -> Potential:
    """Differentiating the lattice internal energy at `state` (`[...] State-`) yields the equilibrium magnetic field
    plane and temperature (`[...] Potential+`) as `temperature * (s - spin_slope)`."""
    shares = (magnet_plane & state) / moment                                  # [...] Scalar+
    # Constant lattice heat capacity makes temperature grow exponentially with the non-spin entropy.
    lattice_entropy = (mv.s & state) - spin_entropy(shares)                   # [...] Scalar+
    temperature = reference * (lattice_entropy / heat_capacity).exp()         # [...] Scalar+
    # Spin entropy decreases with alignment, giving an entropic restoring field proportional to temperature.
    spin_slope = -(shares.arctanh() / moment) * magnet_plane                  # [...] Magnetic+
    return temperature * (mv.s - spin_slope)


shares = mv.scalar(np.linspace(0, largest_share, n_shares)[:, None])          # [n_shares] Scalar+
eos_states = moment * shares * spin + spin_entropy(shares) * entropy          # [n_shares] State-
eos_potentials = potentials(eos_states)                                       # [n_shares] Potential+
render.draw_equation_of_state(eos_states, eos_potentials, moment);

At small ratios of field to temperature the magnetization rises along Curie's linear law on the dashed line, then bends toward saturation and reaches nine tenths of saturation near 2.2 tesla per kelvin.

## Magnetocaloric stiffness and Maxwell relation

Differentiating the internal energy a second time over `State` gives the stiffness map from state increments to potential increments as the sum of two symmetric dyads. Adding energy warms the lattice and scales every potential in proportion to `potentials`, while increasing magnetization steepens the spin-entropy slope against the isothermal spin susceptibility along `magnet_plane`. Because both dyads are symmetric, the adiabatic warming per unit magnetization `stiffness(spin) & entropy` equals the isomagnetization field rise per unit entropy `stiffness(entropy) & spin`, giving the magnetocaloric Maxwell relation directly, and dividing by `stiffness(spin) & spin` gives the adiabatic cooling rate per tesla removed.

In [ ]:
highest_field, bath_temperature = scenarios.highest_field, scenarios.bath_temperature

In [ ]:
def stiffness(state: State) -> Stiffness:
    """The second derivative of internal energy at `state` (`[...] State-`) sums the thermal and spin-ordering dyads
    into a symmetric stiffness map (`[...] Potential+ <- State-`)."""
    shares = (magnet_plane & state) / moment                                  # [...] Scalar+
    salt_potentials = potentials(state)                                       # [...] Potential+
    temperature = salt_potentials & entropy                                   # [...] Scalar+
    thermal_stiffness = salt_potentials * (salt_potentials & State) / (heat_capacity * temperature)   # [...] Potential+ <- State-
    spin_stiffness = temperature / (moment ** 2 * (1 - shares ** 2)) * magnet_plane * (magnet_plane & State)   # [...] Potential+ <- State-
    return thermal_stiffness + spin_stiffness


def equilibrium_state(field: Magnetic, temperature: Scalar) -> State:
    """Spin-half magnetization saturation and logarithmic lattice entropy determine the equilibrium state
    (`[...] State-`) at `field` (`[...] Magnetic+`) and `temperature` (`[...] Scalar+`)."""
    shares = (moment * (field & spin) / temperature).tanh()                   # [...] Scalar+
    entropies = spin_entropy(shares) + heat_capacity * (temperature / reference).log()   # [...] Scalar+
    return moment * shares * spin + entropies * entropy


magnetized_state = equilibrium_state(highest_field * magnet_plane, mv.scalar([bath_temperature]))   # [] State-
magnetized_stiffness = stiffness(magnetized_state)                            # [] Potential+ <- State-
temperature_per_magnetization = magnetized_stiffness(spin) & entropy          # [] Scalar+
field_per_entropy = magnetized_stiffness(entropy) & spin                      # [] Scalar+
cooling_per_field = temperature_per_magnetization / (magnetized_stiffness(spin) & spin)   # [] Scalar+

In [ ]:
print("temperature per magnetization at fixed entropy:  ", temperature_per_magnetization.to_array())
print("field per entropy at fixed magnetization:        ", field_per_entropy.to_array())
print("cooling per tesla removed at fixed entropy (K/T):", cooling_per_field.to_array())

Both cross terms equal the applied field divided by the lattice heat capacity, 1333 kelvin per unit magnetization in these units. Removing one tesla at fixed entropy cools the salt by 0.24 kelvin, slightly below the 0.25 kelvin per tesla of a zero-capacity lattice because heat flowing out of the lattice into the disordering spins slows the temperature drop.

## Ideal refrigeration cycle

An ideal magnetic refrigerator with instantaneous thermal contact traces a rectangle on the temperature-entropy chart between the bath and the cold load. Magnetizing in contact with the bath rejects spin entropy as heat, isolated demagnetization cools the salt at fixed total entropy down to the load temperature, further demagnetization in contact with the load absorbs heat as the spins disorder, and isolated magnetization warms the salt back to the bath temperature. Fixing the highest field at the bath corner and the lowest field at the load corner determines the two cycle entropies, and inverting the spin entropy at the opposite reservoir temperatures fixes the intermediate fields where the isolated legs reach each reservoir. A constant lattice heat capacity is an idealization here, since real crystal capacities fall as the cube of temperature and spin-spin interactions set the millikelvin floor.

In [ ]:
load_temperature, lowest_field, n_halvings = scenarios.load_temperature, scenarios.lowest_field, scenarios.n_halvings
n_line_samples, line_temperatures = scenarios.n_line_samples, scenarios.line_temperatures

In [ ]:
def shares_at_entropy(spin_entropies: Scalar) -> Scalar:
    """Bisection over `n_halvings` intervals inverts the two-level mixing entropy to find the positive magnetization
    shares (`[...] Scalar+`) matching `spin_entropies` (`[...] Scalar+`)."""
    lower_share, upper_share = 0 * spin_entropies, 0 * spin_entropies + 1     # [...] Scalar+, Scalar+
    for _ in range(n_halvings):
        mid_share = (lower_share + upper_share) / 2                           # [...] Scalar+
        above_target = spin_entropy(mid_share) > spin_entropies
        lower_share = where(above_target, mid_share, lower_share)
        upper_share = where(above_target, upper_share, mid_share)
    return (lower_share + upper_share) / 2


held_fields = stack((highest_field * magnet_plane, lowest_field * magnet_plane))   # [2] Magnetic+
held_temperatures = mv.scalar([[bath_temperature], [load_temperature]])       # [2] Scalar+
held_entropies = mv.s & equilibrium_state(held_fields, held_temperatures)     # [2] Scalar+
# Adiabatic legs preserve total entropy, so the spin share absorbs the lattice entropy shift to the other bath.
target_temperatures = mv.scalar([[load_temperature], [bath_temperature]])     # [2] Scalar+
corner_shares = shares_at_entropy(held_entropies - heat_capacity * (target_temperatures / reference).log())   # [2] Scalar+
corner_potentials = potentials(moment * corner_shares * spin + held_entropies * entropy)   # [2] Potential+
cooled_field, warmed_field = (corner_potentials & spin) * magnet_plane        # [] Magnetic+, Magnetic+
corner_fields = stack((warmed_field, held_fields[0], cooled_field, held_fields[1]))   # [4] Magnetic+

high_field_entropy, low_field_entropy = held_entropies                        # [] Scalar+, Scalar+
corner_entropies = stack((low_field_entropy, high_field_entropy, high_field_entropy, low_field_entropy))   # [4] Scalar+
corner_temperatures = mv.scalar([[bath_temperature], [bath_temperature], [load_temperature], [load_temperature]])   # [4] Scalar+
line_states = equilibrium_state(held_fields[:, None], mv.scalar(line_temperatures[..., None]))   # [2, n_line_samples] State-
render.draw_cycle(corner_entropies, corner_temperatures, line_states, potentials(line_states));

The ideal cycle runs counter-clockwise around the rectangle, rejecting heat along the top bath isotherm as the field rises to 4 tesla, cooling down the right isentrope in isolation, absorbing heat along the bottom load isotherm as the field falls to 1 millitesla, and warming up the left isentrope back to the bath temperature. Faint grey curves trace the constant-field states at 4 tesla and 1 millitesla through opposite corners, and the enclosed area equals the magnetic work per spin over one cycle.

## Conduction across the triangulated crystal

Across a finite slab of triangulated sites, each vertex represents a unit-weight point whose lumped dual cell holds one third of the area of its incident triangles. The term ***simplex gradient*** here means the spatial slope of the affine plane that reads one at a triangle's corner and zero along its opposite side, formed by joining the opposite side with `time` and `entropy`, normalizing at the corner, and projecting onto `Plane`. Isotropic lattice conductivity sums dyads `crystal_axes * (Plane & crystal_axes)` over the crystal's two axes on each triangle, and pulling that medium back through the simplex-gradient adjugate `gradient.adjugate()(medium(gradient))` assembles the symmetric heat-conduction map between vertex temperatures.

In [ ]:
topology, coordinates = scenarios.slab(scenarios.n_along, scenarios.n_across)
V, E, F = topology.n_elements
conductivity = scenarios.conductivity

In [ ]:
vertices = (mv(Plane, coordinates) + mv.w).dual().field()                     # Point-[V]
corners = topology.incidence[2, 0] * vertices                                 # [3] Point-[F]
opposite_vertices = topology.off[2, 0] * vertices                             # [2, 3] Point-[F]
# Joining the opposite side with time and entropy and normalizing at the corner yields the affine plane whose
# spatial projection is the corner's simplex gradient.
sides = opposite_vertices[0] & opposite_vertices[1]                           # [3] Antibivector-[F]
gradients = ((sides & time & entropy) / (sides & time & entropy & corners)).cast(Plane)   # [3] Plane+[F]
areas = (corners[0] & corners[1] & corners[2] & time & entropy).norm() / 2    # Scalar+[F]
dual_areas = topology.incidence[2, 0].mean(axis=0).adjugate() * areas         # Scalar+[V]

gradient = (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar   # (Plane+ <- Scalar+)[F <- V]
medium = areas * conductivity * (crystal_axes * (Plane & crystal_axes)).sum(axis=0)   # (Point- <- Plane+)[F]
lattice_conduction = gradient.adjugate()(medium(gradient))                    # (Antiscalar- <- Scalar+)[V <- V]
spin_sites = topology.interior[0]                                             # [V_interior]
render.draw_crystal(topology, vertices, spin_sites, bath_temperature, load_temperature);

Grey edges mark the triangulated slab between the warm red bath bar on the left and the cold blue load bar on the right, both shown retracted while their heat switches are open. Dark dots mark the interior vertices where representative spins are sampled during the cycle.

## Magnetic refrigeration in the crystal

Spins equilibrate rapidly with the applied magnetic field while heat diffuses more slowly across the lattice, so each time step solves the vertex temperatures implicitly using the local compliance `stiffness(states).batch().inverse().field()`. Applying that compliance to `magnet_plane` and `s` separates isothermal magnetocaloric heat release from held-field heat capacity, turning the dual-cell heat balance into one linear solve per step. Starting from thermal equilibrium at the bath temperature, the contact phases sweep the field faster than heat crosses the slab so temperature fronts propagate inward from the linked end before relaxing during the hold. Executing the cycle and rendering the animation below takes about five seconds.

In [ ]:
from_corners, to_corners, phase_progress, conductances, durations, steps_per_frame = scenarios.timeline(coordinates)
n_steps, n_frames = scenarios.n_steps, scenarios.n_frames
links = mv.scalar(conductances[..., None]).field()                            # [n_steps] Scalar+[V]
reservoir_temperatures = mv.scalar(scenarios.reservoirs(coordinates)[:, None]).field()   # Scalar+[V]
draws = mv.scalar(scenarios.spin_draws(V)[:, None]).field()                   # Scalar+[V]
line_ratios = scenarios.line_ratios                                           # [n_line_samples]
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
log_strengths = (corner_fields & spin).log()                                  # [4] Scalar+
fields = (log_strengths[from_corners] * (1 - phase_progress) + log_strengths[to_corners] * phase_progress).exp() * magnet_plane   # [n_steps] Magnetic+


def step(states: State, field: Magnetic, step_links: Scalar, dt: float) -> State:
    """An implicit heat-balance solve over `dt` advances `states` (`State-[V]`) under applied `field` (`[] Magnetic+`)
    and switch conductances `step_links` (`Scalar+[V]`)."""
    salt_potentials = potentials(states)                                      # Potential+[V]
    temperatures = salt_potentials & entropy                                  # Scalar+[V]
    compliance = stiffness(states).batch().inverse().field()                  # (State- <- Potential+)[V]
    field_response = compliance(magnet_plane)                                 # State-[V]
    thermal_response = compliance(mv.s)                                       # State-[V]
    # Magnetizing at fixed temperature expels spin entropy and releases magnetocaloric heat into the dual cells.
    field_shortfall = (field - salt_potentials) & spin                        # Scalar+[V]
    cell_capacities = dual_areas * temperatures * (mv.s & thermal_response)   # Scalar+[V]
    magnetocaloric_heat = dual_areas * temperatures * (mv.s & field_response) * field_shortfall   # Scalar+[V]
    reservoir_coupling = spdiag(step_links * heat * Scalar)                   # (Antiscalar- <- Scalar+)[V <- V]
    heat_balance = spdiag(cell_capacities * heat * Scalar) + dt * (lattice_conduction + reservoir_coupling)   # (Antiscalar- <- Scalar+)[V <- V]
    net_heat = dt * (reservoir_coupling(reservoir_temperatures - temperatures) - lattice_conduction(temperatures)) - magnetocaloric_heat * heat   # Antiscalar-[V]
    temperature_step = heat_balance.solve(net_heat)                           # Scalar+[V]
    return states + field_shortfall * field_response + temperature_step * thermal_response


def run(states: State) -> Iterator[tuple[State, Potential, Magnetic, Scalar]]:
    """Yields vertex states (`State-[V]`), potentials (`Potential+[V]`), applied field (`[] Magnetic+`), and switch
    conductances (`Scalar+[V]`) initially and after each frame's implicit steps."""
    yield states, potentials(states), fields[0], links[0]
    frame_ends = np.cumsum(steps_per_frame)
    for start, end in zip(frame_ends - steps_per_frame, frame_ends):
        for index in range(start, end):
            states = step(states, fields[index], links[index], durations[index])
        yield states, potentials(states), fields[end - 1], links[end - 1]

In [ ]:
def drawn_spins(states: State) -> Spin:
    """Thresholding fixed uniform `draws` against the aligned population fraction of `states` (`State-[V]`) assigns one
    two-level spin (`Spin-[V]`) per vertex."""
    shares = (magnet_plane & states) / moment                                 # Scalar+[V]
    is_aligned = draws < (1 + shares) / 2
    return moment * where(is_aligned, spin, -spin)


def scene_frames(frame_stream: Iterator[tuple[State, Potential, Magnetic, Scalar]]
                 ) -> Iterator[tuple[State, Potential, Magnetic, Scalar, Spin, State, Potential]]:
    """Yields each simulation frame augmented with sampled vertex spins (`Spin-[V]`) and instantaneous constant-field
    states (`[n_line_samples] State-`) and potentials (`[n_line_samples] Potential+`)."""
    for states, salt_potentials, field, step_links in frame_stream:
        field_states = equilibrium_state(field, moment * (field & spin) / line_ratios)   # [n_line_samples] State-
        yield states, salt_potentials, field, step_links, drawn_spins(states), field_states, potentials(field_states)


initial_temperatures = mv.scalar(np.full((V, 1), bath_temperature)).field()   # Scalar+[V]
initial_states = equilibrium_state(corner_fields[0], initial_temperatures)    # State-[V]
frames = show_states(scene_frames(run(initial_states)), render.animate_crystal, topology, vertices, spin_sites,
                     bath_temperature, load_temperature, highest_field, corner_entropies, corner_temperatures,
                     line_states, potentials(line_states), name="paramagnetic_salt_crystal",
                     duration_ms=frame_duration_ms)

Filled dots mark spins aligned with the field and open rings mark spins opposed to it, overlaid on the slab's logarithmic temperature tint from blue through pale yellow to red inside the green wash of the magnet gap. Left and right bars touch the slab whenever the bath or load switch closes, while on the chart the black dots track states along the slab's centreline against the moving green constant-field curve. Spins align from the bath end inward during magnetization, remain ordered during isolated cooling, and disorder from the load end inward as heat enters from the load.

In [ ]:
# checks
# Curie's linear law holds at the smallest non-zero magnetization share.
curie_slope = moment * shares[1] / ((eos_potentials[1] & spin) / (eos_potentials[1] & entropy))
assert ((curie_slope - moment ** 2).abs() < 1e-3 * moment ** 2).all()
# Symmetric stiffness dyads enforce the magnetocaloric Maxwell relation.
assert ((temperature_per_magnetization - field_per_entropy).abs() < 1e-9 * highest_field / heat_capacity).all()
assert ((temperature_per_magnetization - highest_field / heat_capacity).abs() < 1e-9 * highest_field / heat_capacity).all()
# Adiabatic corner solutions recover the opposite reservoir temperatures at fixed total entropy.
assert (((corner_potentials & entropy) - target_temperatures).abs() < 1e-3 * target_temperatures).all()
# Simplex gradients sum to zero on every triangle and lumped dual areas conserve total slab area.
assert (gradients.batch().sum(axis=0).norm() < 1e-9).all()
assert ((dual_areas.sites.sum() - areas.sites.sum()).abs() < 1e-9).all()
# Cycle entropy and temperature invariants hold across the bath, adiabatic cooling, and load phases.
frame_entropies = stack([
    (dual_areas * (mv.s & frame_states)).sites.sum() for frame_states, *_ in frames
])                                                                            # [n_frames + 1] Scalar+
cumulative_frames = np.cumsum(scenarios.n_phase_frames)
bath_end, demagnetized_end, load_end = cumulative_frames[1], cumulative_frames[2], cumulative_frames[4]
assert (frame_entropies[bath_end] < frame_entropies[0]).all()
assert ((frame_entropies[demagnetized_end] - frame_entropies[bath_end]).abs() < 1e-2 * frame_entropies[bath_end]).all()
bath_temperatures = (frames[bath_end][1] & entropy).batch()                   # [V] Scalar+
demagnetized_temperatures = (frames[demagnetized_end][1] & entropy).batch()   # [V] Scalar+
load_temperatures = (frames[load_end][1] & entropy).batch()                   # [V] Scalar+
assert (demagnetized_temperatures < bath_temperatures.min() / 50).all()
assert (frame_entropies[load_end] > frame_entropies[demagnetized_end]).all()
assert ((load_temperatures - load_temperature).abs() < 0.1 * load_temperature).all()
assert ((frame_entropies[-1] - frame_entropies[0]).abs() < 1e-2 * frame_entropies[0]).all()